# Notebook 04 · Freeze strata, primary selections, and reserve order

Fresh-run release. Follow the input and review gates in RUN_ORDER.md.

In [1]:
import csv
import hashlib
import io
import json
import os
import re
import zipfile
from pathlib import Path, PurePosixPath

INPUT = Path(os.environ.get("OPENAPI_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("OPENAPI_WORK", "/kaggle/working"))
DESIGN = "openapi-fresh-pilot-v1"


def sha(data):
    return hashlib.sha256(data).hexdigest()


def file_sha(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()


def encode(data):
    return (json.dumps(data, sort_keys=True, ensure_ascii=False, separators=(",", ":"),
                       allow_nan=False) + "\n").encode()


def load(path):
    return json.loads(path.read_bytes())


def records(path):
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def csv_records(path):
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def atomic(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(data)
    tmp.replace(path)


def jsonl(path, rows):
    atomic(path, b"".join(map(encode, rows)))


def csv_write(path, fields, rows):
    output = io.StringIO(newline="")
    writer = csv.DictWriter(output, fieldnames=fields, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    atomic(path, output.getvalue().encode())


def safe_path(name):
    p = PurePosixPath(name)
    if p.is_absolute() or ".." in p.parts or "\\" in name or p.as_posix() != name:
        raise ValueError(f"Unsafe manifest path: {name}")
    return Path(*p.parts)


import hashlib
import json
import re
from pathlib import PurePosixPath


ARTIFACT_SCHEMA = "openapi-artifact-v1"
PRODUCER_NOTEBOOKS = {
    "00": "00_protocol_and_runtime",
    "01": "01_corpus_preparation_and_eligibility",
    "02": "02_operation_slicing_and_prose_measurement",
    "03": "03_pair_and_evaluator_feasibility",
    "04": "04_strata_and_pilot_selection",
    "05": "05_oracle_build_and_review",
    "06": "06_build_corrected_evaluator_and_generation_tasks",
    "07": "07_pilot_generation",
    "08": "08_pilot_execution_and_review",
}


def provenance_digest(value):
    data = json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=False, allow_nan=False).encode("utf-8")
    return hashlib.sha256(data).hexdigest()


def artifact_path_for(record):
    paths = {record[key] for key in ("artifact_path", "relative_path", "path")
             if key in record}
    if len(paths) != 1:
        raise RuntimeError("Missing or inconsistent artifact paths")
    name = paths.pop()
    path = PurePosixPath(name)
    if not name or name == "." or path.is_absolute() or ".." in path.parts or "\\" in name or path.as_posix() != name:
        raise RuntimeError(f"Invalid artifact path: {name!r}")
    return name


def manifest_digest_valid(value):
    return isinstance(value, str) and re.fullmatch(r"[0-9a-f]{64}", value) is not None


def register_artifact_provenance(manifest):
    stage = manifest["stage"]
    artifacts = manifest["artifacts"]
    if "upstream_manifest_sha256" not in manifest:
        if stage != "00":
            raise RuntimeError(f"Stage {stage} has no upstream identity")
        manifest["upstream_manifest_sha256"] = {}
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest.get("configuration_hash")
    if config_hash is None:
        config_hash = provenance_digest({artifact_path_for(row): row["sha256"]
                                         for row in artifacts
                                         if artifact_path_for(row).startswith("config/")})
    manifest["configuration_hash"] = config_hash
    manifest["artifact_record_schema"] = ARTIFACT_SCHEMA
    upstream_hash = provenance_digest(links)
    for row in artifacts:
        row.update(artifact_path=artifact_path_for(row), schema_version=ARTIFACT_SCHEMA,
                   producer_notebook=PRODUCER_NOTEBOOKS[stage], config_hash=config_hash,
                   upstream_manifest_hash=upstream_hash)
    verify_artifact_provenance(manifest)


def verify_artifact_provenance(manifest):
    stage = manifest["stage"]
    if manifest.get("artifact_record_schema") != ARTIFACT_SCHEMA:
        raise RuntimeError(f"Stage {stage} requires a corrected provenance export")
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest["configuration_hash"]
    if (not isinstance(links, dict) or bool(links) != (stage != "00")
            or any(key not in PRODUCER_NOTEBOOKS or key >= stage or not manifest_digest_valid(value)
                   for key, value in links.items()) or not manifest_digest_valid(config_hash)):
        raise RuntimeError(f"Invalid Stage {stage} provenance")
    expected = (ARTIFACT_SCHEMA, PRODUCER_NOTEBOOKS[stage], config_hash, provenance_digest(links))
    seen = set()
    artifacts = manifest["artifacts"]
    if not artifacts or len(artifacts) != manifest["artifact_count"]:
        raise RuntimeError(f"Invalid Stage {stage} artifact count")
    for row in artifacts:
        name = artifact_path_for(row)
        actual = tuple(row.get(key) for key in
                       ("schema_version", "producer_notebook", "config_hash", "upstream_manifest_hash"))
        if (row.get("artifact_path") != name or name in seen or actual != expected
                or not manifest_digest_valid(row.get("sha256"))
                or type(row.get("bytes")) is not int or row["bytes"] < 0):
            raise RuntimeError(f"Invalid Stage {stage} artifact provenance: {name}")
        seen.add(name)


def stage_input(stage, *, fresh=True):
    matches = []
    for external in sorted(INPUT.rglob(f"stage_{stage}_manifest.json")):
        sidecar = load(external)
        if fresh and sidecar.get("design_id") != DESIGN:
            continue
        for manifest in external.parent.rglob(f"registries/{stage}_manifest.json"):
            matches.append((external, manifest))
    if len(matches) != 1:
        raise RuntimeError(f"Attach one expanded Stage {stage} release. Found {len(matches)}.")
    external, internal = matches[0]
    sidecar, manifest = load(external), load(internal)
    expected = sidecar.get("internal_manifest_sha256") or sidecar.get("internal_manifest", {}).get("sha256")
    if expected != file_sha(internal):
        raise RuntimeError(f"Stage {stage} manifest digest mismatch.")
    if (manifest.get("stage_status", manifest.get("status")) != "PASS"
            or sidecar.get("stage_status", sidecar.get("status")) != "PASS"):
        raise RuntimeError(f"Stage {stage} has no passing handoff.")
    verify_artifact_provenance(manifest)
    root = internal.parent.parent
    paths = {}
    for item in manifest["artifacts"]:
        name = item.get("relative_path") or item.get("artifact_path") or item.get("path")
        path = root / safe_path(name)
        if name in paths or path.is_symlink() or not path.is_file() or path.stat().st_size != item["bytes"] or file_sha(path) != item["sha256"]:
            raise RuntimeError(f"Stage {stage} artifact mismatch: {name}")
        paths[name] = item["sha256"]
    if len(paths) != manifest["artifact_count"]:
        raise RuntimeError(f"Stage {stage} artifact count mismatch.")
    return root, manifest, expected, paths


def pack(stage, root, metadata):
    output = WORK / f"stage_{stage}_fresh_dataset"
    output.mkdir(parents=True, exist_ok=True)
    manifest_path = root / f"registries/{stage}_manifest.json"
    files = sorted(p for p in root.rglob("*") if p.is_file() and p != manifest_path
                   and "__pycache__" not in p.parts and not p.name.endswith(".tmp"))
    artifacts = [{"relative_path": p.relative_to(root).as_posix(), "bytes": p.stat().st_size,
                  "sha256": file_sha(p)} for p in files]
    manifest = {"stage": stage, "status": "PASS", "design_id": DESIGN,
                "artifact_count": len(artifacts), "artifacts": artifacts, **metadata}
    register_artifact_provenance(manifest)
    atomic(manifest_path, encode(manifest))
    archive = output / f"stage_{stage}_fresh.zip"
    tmp = archive.with_suffix(".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=4) as z:
        for p in [*files, manifest_path]:
            z.write(p, "prose_operation_experiment/" + p.relative_to(root).as_posix())
    tmp.replace(archive)
    external = {k: v for k, v in manifest.items() if k != "artifacts"}
    external.update(archive=archive.name, archive_sha256=file_sha(archive),
                    archive_bytes=archive.stat().st_size, internal_manifest_sha256=file_sha(manifest_path))
    atomic(output / f"stage_{stage}_manifest.json", encode(external))
    atomic(output / f"stage_{stage}_checksums.sha256",
           f'{external["archive_sha256"]}  {archive.name}\n'.encode())
    atomic(output / "README.md", f"# Stage {stage} fresh pilot\n\nDesign {DESIGN}.\n".encode())
    print(f"Stage {stage} PASS. Publish {output}")
    return external


from collections import Counter, defaultdict
from fractions import Fraction
import shutil

import numpy as np

stage03, manifest03, hash03, artifacts03 = stage_input("03")
project = WORK / "fresh_release/stage_04/prose_operation_experiment"
frame = list(records(stage03 / "data/inventory/eligible_operations.jsonl"))
if len(frame) != len({r["op_id"] for r in frame}) or len(frame) < 100:
    raise RuntimeError("Invalid Stage 03 eligible frame.")
if any(not row["family_id"] or not row["oracle_sha256"] for row in frame):
    raise RuntimeError("Sampling requires verified families and shared-builder feasibility.")
token_counts = Counter(int(row["prose_tokens_L0"]) for row in frame)
boundaries = []
running = 0
for token in sorted(token_counts)[:-1]:
    running += token_counts[token]
    boundaries.append((token, running))
if len(boundaries) < 2:
    raise RuntimeError("At least three distinct L0 prose scores are required.")
total = len(frame)
c1, _ = min(boundaries[:-1], key=lambda item: (abs(3 * item[1] - total), item[0]))
c2, _ = min((item for item in boundaries if item[0] > c1),
            key=lambda item: (abs(3 * item[1] - 2 * total), item[0]))
groups = defaultdict(list)
for row in frame:
    score = int(row["prose_tokens_L0"])
    group = "low" if score <= c1 else "medium" if score <= c2 else "high"
    groups[group].append({**row, "stratum": group})
order = ("low", "medium", "high")
targets = {h: Fraction(100 * len(groups[h]), total) for h in order}
allocation = {h: int(targets[h]) for h in order}
for h in sorted(order, key=lambda h: (-(targets[h] - allocation[h]), order.index(h)))[:100 - sum(allocation.values())]:
    allocation[h] += 1
if any(not 0 < allocation[h] <= len(groups[h]) for h in order):
    raise RuntimeError("The frozen allocation requires positive, feasible strata.")
design = {"seed": 20261103, "generator": "numpy.PCG64", "numpy_version": np.__version__,
          "allocation": "Hamilton, low/medium/high tie order",
          "boundaries": {"c1": c1, "c2": c2, "rule": "closest cumulative empirical thirds; lower cut wins ties"},
          "reserve_rule": "one full uniform permutation per stratum; first n_h are provisional primary selections",
          "failure_policy": "only evidence-backed eligibility exclusions before any generation; rebuild Stage 03 for systemic builder failures",
          "final_probability": "n_h divided by final eligible N_h after enumerating every applicable eligibility exclusion",
          "precision": "provisional 100 pairs; family clustering can reduce precision"}
atomic(project / "config/sampling_design.json", encode(design))
rng = np.random.Generator(np.random.PCG64(design["seed"]))
ranked, selected, strata = [], [], []
for h in order:
    pool = sorted(groups[h], key=lambda row: row["op_id"])
    n = allocation[h]
    strata.append({"stratum": h, "N_h": len(pool), "W_h": len(pool) / total, "n_h": n,
                   "selection_probability": n / len(pool), "sampling_weight": len(pool) / n,
                   "lower_bound": min(int(r["prose_tokens_L0"]) for r in pool),
                   "upper_bound": max(int(r["prose_tokens_L0"]) for r in pool)})
    for rank, index in enumerate(rng.permutation(len(pool)), 1):
        row = {**pool[index], "selection_rank": rank, "inclusion_probability": n / len(pool)}
        ranked.append(row)
        if rank <= n:
            selected.append(row)
jsonl(project / "partitions/pilot/ranked_frame.jsonl", ranked)
selection_fields = ["op_id", "pair_id", "spec_id", "family_id", "stratum", "prose_tokens_L0",
                    "selection_rank", "inclusion_probability", "L0_hash", "L1_hash"]
def selection_row(row):
    return {**{k: row[k] for k in selection_fields[:-2]},
            **{f"{condition}_hash": row[f"{condition}_sha256"] for condition in ("L0", "L1")}}
csv_write(project / "partitions/pilot/selected.csv", selection_fields, map(selection_row, selected))
primary_ids = {r["op_id"] for r in selected}
csv_write(project / "partitions/pilot/remaining_pool.csv", selection_fields,
          (selection_row(r) for r in ranked if r["op_id"] not in primary_ids))
csv_write(project / "partitions/pilot/stratum_sizes.csv", list(strata[0]), strata)
for row in selected:
    for condition in ("L0", "L1"):
        name = f"data/slices/{condition}/{row['op_id']}.json"
        source = stage03 / name
        if artifacts03[name] != row[f"{condition}_sha256"]:
            raise RuntimeError("Selected pair does not match the eligible frame.")
        dest = project / f"partitions/pilot/inputs/{condition}/{row['op_id']}.json"
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, dest)
family_sizes = Counter(row["family_id"] for row in selected)
atomic(project / "reports/audit/family_dependence.json", encode({
    "distinct_families": len(family_sizes), "largest_family": max(family_sizes.values()),
    "concentration_m2_over_m": sum(m * m for m in family_sizes.values()) / len(selected),
    "design_effect_sensitivity": {str(rho): 1 + rho * (sum(m * m for m in family_sizes.values()) / len(selected) - 1)
                                  for rho in (0.0, 0.1, 0.25, 0.5, 1.0)},
    "allocation": allocation, "provisional_target": 100,
}))
pack("04", project, {"upstream_manifest_sha256": {"00": manifest03["upstream_manifest_sha256"]["00"], "03": hash03},
                     "checkpoint_a_reached": True, "may_begin_notebook_05": True, "operation_count": 100,
                     "sampling_design_sha256": file_sha(project / "config/sampling_design.json")})



Stage 04 PASS. Publish /kaggle/working/stage_04_fresh_dataset


{'stage': '04',
 'status': 'PASS',
 'design_id': 'openapi-fresh-pilot-v1',
 'artifact_count': 206,
 'upstream_manifest_sha256': {'00': 'a064f3bb9814b3545ed81984c5a5bcd5a5ce236ce621f2fd0d039950efafc431',
  '03': '74550c1645772a46e668cbbc4024fb36728cff14a27781694e851483251a9272'},
 'checkpoint_a_reached': True,
 'may_begin_notebook_05': True,
 'operation_count': 100,
 'sampling_design_sha256': '34a370cea07f57dff31b432bb31569e223efae70721ee89eaf6fd53e80ba7897',
 'configuration_hash': 'bb4422d68b977e009fca66961e18173a98be53f855a589708425280609af7bd1',
 'artifact_record_schema': 'openapi-artifact-v1',
 'archive': 'stage_04_fresh.zip',
 'archive_sha256': '9be899ba69455fc426d4e844180e3b2842f6191a956edb50502bdf4b85f4c3db',
 'archive_bytes': 7961394,
 'internal_manifest_sha256': '74efa268b726abf8badbc408d909608c59ab1bda84f5e2037f50cfd7b0692e8d'}